<a href="https://colab.research.google.com/github/virastyfitri-lab/tender-evaluation-project/blob/main/tender_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Daftar file dari 3 penilai berbeda
files = ['penilaian tender_1.xlsx', 'penilaian tender_2.xlsx', 'penilaian tender_3.xlsx']

all_evaluations = []

# 1. Membaca data dari masing-masing file penilai
for f in files:
    df_raw = pd.read_excel(f, sheet_name='Sheet1', header=None)
    header_df = df_raw.iloc[:8].copy()
    data_df = df_raw.iloc[8:].copy().reset_index(drop=True)
    all_evaluations.append(data_df)

print(f"Berhasil membaca {len(files)} file penilai.")

# Gunakan file pertama sebagai template struktur dasar
base_df = all_evaluations[0].copy()

# Indeks kolom penting (berbasis 0)
col_name = 1                 # Nama Perusahaan
col_identity_status = 12     # IDENTITY EVALUATION STATUS
col_offer_status = 15        # OFFER VALUES STATUS
col_offer_score = 16         # OFFER VALUE SCORE
col_validity_status = 19     # VALIDITY PERIOD STATUS
col_workplan_status = 23     # WORK PLAN EVALUATION STATUS
col_tech_status = 27         # TECHNICAL EVALUATION STATUS
col_schedule_status = 30     # SCHEDULE EVALUATION STATUS
col_past_perf_status = 33    # PAST PERFORMANCE EVALUATION STATUS
col_past_perf_score = 32     # NUMBER OF SIMILAR PROJECTS

col_final_status = 34        # FINAL COMPLIANCE STATUS
col_final_score = 35         # FINAL WEIGHTED SCORE

# 2. Melakukan Agregasi dan Konsensus untuk setiap vendor
summary_results = []

for i in range(len(base_df)):
    vendor_name = base_df.iloc[i, col_name]

    pass_counts = 0
    total_scores = []

    # Periksa penilaian dari ke-3 penilai
    for eval_df in all_evaluations:
        row = eval_df.iloc[i]

        # Cek apakah semua kriteria Pass di penilai ini
        is_id = str(row[col_identity_status]).strip().upper() == 'PASS'
        is_off = str(row[col_offer_status]).strip().upper() == 'PASS'
        is_val = str(row[col_validity_status]).strip().upper() == 'PASS'
        is_wrk = str(row[col_workplan_status]).strip().upper() == 'PASS'
        is_tch = str(row[col_tech_status]).strip().upper() == 'PASS'
        is_sch = str(row[col_schedule_status]).strip().upper() == 'PASS'
        is_prf = str(row[col_past_perf_status]).strip().upper() == 'PASS'

        evaluator_passed = (is_id and is_off and is_val and is_wrk and is_tch and is_sch and is_prf)

        if evaluator_passed:
            pass_counts += 1

        # Hitung skor dari penilai ini
        try:
            p_score = float(row[col_offer_score]) if pd.notna(row[col_offer_score]) else 0
            pr_cnt = float(row[col_past_perf_score]) if pd.notna(row[col_past_perf_score]) else 0
            exp_score = min((pr_cnt / 5.0) * 100, 100)

            w_score = (0.70 * p_score) + (0.30 * exp_score)
            total_scores.append(w_score if evaluator_passed else 0)
        except:
            total_scores.append(0)

    # Aturan Konsensus: Minimal 2 dari 3 penilai harus meluluskan vendor tersebut
    is_consensus_qualified = (pass_counts >= 2)

    final_compliance = 'QUALIFIED' if is_consensus_qualified else 'DISQUALIFIED'
    avg_score = round(np.mean(total_scores), 2) if is_consensus_qualified else 0.0

    # Masukkan ke baris template dasar
    base_df.iloc[i, col_final_status] = final_compliance
    base_df.iloc[i, col_final_score] = avg_score

    summary_results.append({
        'Nama Perusahaan': vendor_name,
        'Lolos (Dari 3 Penilai)': f"{pass_counts}/3 Penilai",
        'Status Akhir': final_compliance,
        'Skor Rata-Rata': avg_score
    })

# 3. Simpan Hasil Konsensus ke File Excel Baru
output_file = 'hasil_konsensus_tender.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    base_df.to_excel(writer, sheet_name='Sheet1', index=False, header=False)

print(" konsensus multi-penilai selesai!")
df_summary = pd.DataFrame(summary_results)
print(df_summary.to_string(index=False))

Berhasil membaca 3 file penilai.
 konsensus multi-penilai selesai!
    Nama Perusahaan Lolos (Dari 3 Penilai) Status Akhir  Skor Rata-Rata
       PT Maju Jaya            3/3 Penilai    QUALIFIED           30.35
PT Berkah Sejahtera            0/3 Penilai DISQUALIFIED            0.00
    PT Inti Cendana            0/3 Penilai DISQUALIFIED            0.00
      CV Bumiputera            0/3 Penilai DISQUALIFIED            0.00
   CV Angin Bahagia            0/3 Penilai DISQUALIFIED            0.00
    PT Sumber Waras            3/3 Penilai    QUALIFIED           33.01
    PT Berkah Ilahi            1/3 Penilai DISQUALIFIED            0.00
     CV Duren Mekar            0/3 Penilai DISQUALIFIED            0.00
    CV Persahabatan            1/3 Penilai DISQUALIFIED            0.00
    PT Bangun Karsa            0/3 Penilai DISQUALIFIED            0.00
 PT Sejahtera Abadi            0/3 Penilai DISQUALIFIED            0.00
   PT Damai Sentosa            0/3 Penilai DISQUALIFIED            0.